In [1]:
!curl -O https://raw.githubusercontent.com/esensato/ssa-2026-01/main/imoveis.csv

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  10936 100  10936   0      0  37517      0                              0
100  10936 100  10936   0      0  37484      0                              0
100  10936 100  10936   0      0  37441      0                              0


In [2]:
import pandas as pd

# Leitura do arquivo CSV
# Considera separador ";" e trata valores nulos comuns
imoveis = pd.read_csv("imoveis.csv", sep=";", encoding="utf-8", na_values=["", "-", "null", "NULL", "N/A"])

# Visualização inicial
display(imoveis.shape)
display(imoveis.head())
imoveis.info()

(100, 15)

,id_imovel,id_proprietario,nome_proprietario,cpf,endereco,bairro,cidade,data_anuncio,tipo_imovel,metragem,quartos,banheiros,valor,valor_iptu,sistema_origem
0,1,P003,ANA PEREIRA,323.338.617-87,Rua W 167,Vila Mariana,Sao Paulo,09-01-2023,casa,165,3.0,1,729895,1212,Planilha
1,2,P003,Carla Dias,51025702129,Rua S 50,centro,Sao Paulo,05/01/2023,Loft,150,3.0,1,689004,3317,Planilha
2,3,P011,Ana Pereira,350.267.573-58,Rua V 84,Centro,Sao Paulo,2023-01-15,Loft,123m2,3.0,4,"1.790.113,00","5808,00",API
3,4,P009,CARLOS LIMA,64045862384,Rua H 36,Itaim,Sao Paulo,2023-01-31,Loft,NaN,1.0,2,2393927,"4258,00",Planilha
4,5,P008,Carla Dias,24345944431,Rua N 41,Itaim,Sao Paulo,2023/01/13,casa,NaN,2.0,1,1354507,594,API


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_imovel          100 non-null    int64  
 1   id_proprietario    100 non-null    str    
 2   nome_proprietario  100 non-null    str    
 3   cpf                100 non-null    str    
 4   endereco           100 non-null    str    
 5   bairro             100 non-null    str    
 6   cidade             100 non-null    str    
 7   data_anuncio       100 non-null    str    
 8   tipo_imovel        100 non-null    str    
 9   metragem           65 non-null     str    
 10  quartos            82 non-null     float64
 11  banheiros          100 non-null    int64  
 12  valor              100 non-null    str    
 13  valor_iptu         100 non-null    str    
 14  sistema_origem     58 non-null     str    
dtypes: float64(1), int64(2), str(12)
memory usage: 11.8 KB


In [ ]:
# 1. Remover espaços extras e padronizar maiúsculas/minúsculas
for col in ["nome_proprietario", "endereco", "bairro", "cidade", "tipo_imovel", "sistema_origem"]:
    imoveis[col] = imoveis[col].astype(str).str.strip().str.title()

In [ ]:
# 2. Padronizar CPF (remover pontos e traços)
imoveis["cpf"] = imoveis["cpf"].astype(str).str.replace(".", "").str.replace("-", "")

In [ ]:
# 3. Padronizar datas
def parse_data(valor):
    formatos = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y", "%Y.%m.%d"]
    for f in formatos:
        try:
            return pd.to_datetime(valor, format=f)
        except:
            pass
    return pd.NaT
imoveis["data_anuncio"] = imoveis["data_anuncio"].apply(parse_data)

In [ ]:
# 4. Converter valores numéricos (metragem, valor, valor_iptu)
def parse_number(x):
    if pd.isnull(x):
        return None
    x = str(x).replace("m2", "").replace(".", "").replace(",", ".").strip()
    try:
        return float(x)
    except:
        return None
imoveis["metragem"] = imoveis["metragem"].apply(parse_number)
imoveis["valor"] = imoveis["valor"].apply(parse_number)
imoveis["valor_iptu"] = imoveis["valor_iptu"].apply(parse_number)

In [ ]:
# 5. Converter quartos e banheiros para inteiro
for col in ["quartos", "banheiros"]:
    imoveis[col] = pd.to_numeric(imoveis[col], errors="coerce").fillna(0).astype(int)

In [ ]:
# 6. Remover duplicidades
imoveis = imoveis.drop_duplicates()

In [ ]:
# 7. Tratar valores ausentes (exemplo: preencher metragem ausente com mediana)
imoveis["metragem"] = imoveis["metragem"].fillna(imoveis["metragem"].median())

In [ ]:

imoveis.head()

In [ ]:
# Exportar o dataset limpo
imoveis.to_csv("imoveis_limpo.csv", index=False)
print("Arquivo 'imoveis_limpo.csv' gerado com sucesso!")